In [1]:
import numpy as np
from PIL import Image
import os
from random import shuffle
from typing import Tuple, List, Optional

class MLPBackpropagation:
    def __init__(self, input_size: int, hidden_size: int = 4, output_size: int = 3, learning_rate: float = 0.1) -> None:
        """Инициализаия многослойного перцептрона"""
        self.W: np.ndarray = np.random.randn(input_size, hidden_size) * 0.01
        self.V: np.ndarray = np.random.randn(hidden_size, output_size) * 0.01
        self.learning_rate: float = learning_rate
        self.mean: Optional[np.ndarray] = None
        self.std: Optional[np.ndarray] = None
        
    def softmax(self, x: np.ndarray) -> np.ndarray:
        """Функция активации softmax"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Сигмойдная функция активации"""
        return 1 / (1 + np.exp(-x))
    
    def sigmoid_derivative(self, x: np.ndarray) -> np.ndarray:
        """Производная сигмойдной функции"""
        return x * (1 - x)
    
    def normalize(self, X: np.ndarray) -> np.ndarray:
        """Нормализация входных данных"""
        if self.mean is None:
            self.mean = X.mean(axis=0)
            self.std = X.std(axis=0) + 1e-6
        return (X - self.mean) / self.std
    
    def forward(self, X: np.ndarray) -> np.ndarray:
        """Прямой проход через сеть"""
        self.hidden_input = np.dot(X, self.W)
        self.hidden_output = self.sigmoid(self.hidden_input)
        self.output_input = np.dot(self.hidden_output, self.V)
        self.output = self.softmax(self.output_input) 
        return self.output
    
    def backward(self, X: np.ndarray, y: np.ndarray, output: np.ndarray) -> None:
        """Обратное распространение ошибки"""
        output_error = output - y
        hidden_error = output_error.dot(self.V.T) * self.sigmoid_derivative(self.hidden_output)
        
        self.V -= self.hidden_output.T.dot(output_error) * self.learning_rate
        self.W -= X.T.dot(hidden_error) * self.learning_rate
    
    def fit(self, X_train: np.ndarray, y_train: np.ndarray, epochs: int = 1000, min_error: float = 0.01) -> None:
        """Обучение модели"""
        X_train_norm = self.normalize(X_train)
        
        for epoch in range(epochs):
            output = self.forward(X_train_norm)
            self.backward(X_train_norm, y_train, output)
            
            # Вычисление cross-entropy loss
            loss = -np.mean(y_train * np.log(output + 1e-7))
            
            if epoch % 100 == 0:
                print(f"Эпоха {epoch}, Loss: {loss:.4f}")
                
            if loss < min_error:
                print(f"Обучение завершено на эпохе {epoch}: loss < {min_error}")
                break
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Предсказание классов для входных данных"""
        X_norm = (X - self.mean) / self.std
        output = self.forward(X_norm)
        return np.argmax(output, axis=1)  # Возвращаем индекс класса с максимальной вероятностью
    
    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray) -> float:
        """Оценка точности модели на тестовых данных"""
        predictions = self.predict(X_test)
        true_labels = np.argmax(y_test, axis=1)
        accuracy = np.mean(predictions == true_labels)
        return accuracy

def extract_features(image_path: str) -> np.ndarray:
    """Извлечение признаков изображения"""
    try:
        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img)
        brightness = np.mean(img_array) / 255.0
        green_ratio = np.mean(img_array[:, :, 1]) / 255.0
        gray_img = img.convert('L')
        contrast = np.std(np.array(gray_img)) / 255.0
        return np.array([brightness, green_ratio, contrast])
    except Exception as e:
        print(f"Ошибка обработки изображения {image_path}: {str(e)}")
        return np.zeros(3)

def load_dataset(data_dir: str, test_size: float = 0.2) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Загрузка и разделение датасета на обучающую и тестовую выборки"""
    X: List[np.ndarray] = []
    y: List[np.ndarray] = []
    class_names = ['forest', 'desert']
    
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        for filename in os.listdir(class_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_dir, filename)
                features = extract_features(img_path)
                X.append(features)
                # One-hot encoding для меток
                one_hot = np.zeros(len(class_names))
                one_hot[label] = 1
                y.append(one_hot)
    
    combined = list(zip(X, y))
    shuffle(combined)
    X, y = zip(*combined)
    
    split_idx = int(len(X) * (1 - test_size))
    X_train = np.array(X[:split_idx])
    y_train = np.array(y[:split_idx])
    X_test = np.array(X[split_idx:])
    y_test = np.array(y[split_idx:])
    
    return X_train, y_train, X_test, y_test

In [6]:
# Конфигурация
data_dir = "origins/data"

# Разделение на train/test
X_train, y_train, X_test, y_test = load_dataset(data_dir)

# Создание и обучение перцептрона
mlp = MLPBackpropagation(input_size=3, hidden_size=4, output_size=2, learning_rate=0.1)
mlp.fit(X_train, y_train, epochs=10000)

# Оценка точности
train_accuracy = mlp.evaluate(X_train, y_train)
test_accuracy = mlp.evaluate(X_test, y_test)
print(f"\nТочность на обучающей выборке: {train_accuracy*100:.2f}%")
print(f"Точность на тестовой выборке: {test_accuracy*100:.2f}%")

Эпоха 0, Loss: 0.3465
Эпоха 100, Loss: 0.3082
Эпоха 200, Loss: 0.3866
Эпоха 300, Loss: 0.1551
Эпоха 400, Loss: 0.2675
Эпоха 500, Loss: 0.1399
Эпоха 600, Loss: 0.1280
Эпоха 700, Loss: 0.6120
Эпоха 800, Loss: 0.3676
Эпоха 900, Loss: 0.5892
Эпоха 1000, Loss: 0.1735
Эпоха 1100, Loss: 0.1341
Эпоха 1200, Loss: 0.1925
Эпоха 1300, Loss: 0.1090
Эпоха 1400, Loss: 0.2751
Эпоха 1500, Loss: 0.3632
Эпоха 1600, Loss: 0.2106
Эпоха 1700, Loss: 0.2099
Эпоха 1800, Loss: 0.1370
Эпоха 1900, Loss: 0.2394
Эпоха 2000, Loss: 0.4456
Эпоха 2100, Loss: 0.2070
Эпоха 2200, Loss: 0.1239
Эпоха 2300, Loss: 0.2555
Эпоха 2400, Loss: 0.2387
Эпоха 2500, Loss: 0.3099
Эпоха 2600, Loss: 0.3228
Эпоха 2700, Loss: 0.1168
Эпоха 2800, Loss: 0.1169
Эпоха 2900, Loss: 0.2105
Эпоха 3000, Loss: 0.1725
Эпоха 3100, Loss: 0.1072
Эпоха 3200, Loss: 0.3908
Эпоха 3300, Loss: 0.1350
Эпоха 3400, Loss: 0.1061
Эпоха 3500, Loss: 0.3724
Эпоха 3600, Loss: 0.4903
Эпоха 3700, Loss: 0.0954
Эпоха 3800, Loss: 0.1529
Эпоха 3900, Loss: 0.2328
Эпоха 4000, 